In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [2]:
filepath = '/content/retail_data_synthetic_50k.xlsx'
df = pd.read_excel(filepath)

### 1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Transaction_ID     50000 non-null  object        
 1   Customer_ID        50000 non-null  object        
 2   Gender             50000 non-null  object        
 3   Age                50000 non-null  int64         
 4   Category           50000 non-null  object        
 5   Quantity           50000 non-null  int64         
 6   Unit_Price         50000 non-null  float64       
 7   Discount           50000 non-null  float64       
 8   Date               50000 non-null  datetime64[ns]
 9   Store_Region       50000 non-null  object        
 10  Online_Or_Offline  50000 non-null  object        
 11  Payment_Method     50000 non-null  object        
 12  Total_Amount       50000 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(7)
memory 

In [5]:
df.head()

,Transaction_ID,Customer_ID,Gender,Age,Category,Quantity,Unit_Price,Discount,Date,Store_Region,Online_Or_Offline,Payment_Method,Total_Amount
0,TXN-00000,CUST-1127,Female,60,Furniture,9,202.12,0.26,2023-02-12,North,Online,Digital Wallet,1346.12
1,TXN-00001,CUST-1460,Male,30,Beauty,4,75.03,0.05,2021-12-04,West,Online,Digital Wallet,285.11
2,TXN-00002,CUST-0861,Male,52,Clothing,1,374.66,0.15,2020-10-21,South,Online,Cash,318.46
3,TXN-00003,CUST-1295,Non-binary,38,Electronics,7,71.98,0.30,2020-04-26,East,Online,Credit Card,352.70
4,TXN-00004,CUST-1131,Male,47,Grocery,2,169.25,0.02,2022-06-03,West,Online,Cash,331.73


In [6]:
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()
df['IsWeekend'] = df['Date'].dt.dayofweek >= 5

In [7]:
leakage_cols = ['Quantity', 'Unit_Price', 'Discount', 'Transaction_ID', 'Customer_ID', 'Date']
target = 'Total_Amount'

In [8]:
X = df.drop(columns=leakage_cols + [target])
y = df[target]

In [19]:
df['Avg_Spend_Category'] = df.groupby('Category')['Total_Amount'].transform('mean')
df['Avg_Spend_Region'] = df.groupby('Store_Region')['Total_Amount'].transform('mean')
df['Avg_Spend_Method'] = df.groupby('Payment_Method')['Total_Amount'].transform('mean')

In [20]:
X = pd.concat([X, df[['Avg_Spend_Category', 'Avg_Spend_Region', 'Avg_Spend_Method']]], axis=1)

In [21]:
cat_cols = ['Gender', 'Category', 'Store_Region', 'Online_Or_Offline',
            'Payment_Method', 'DayOfWeek', 'Month', 'IsWeekend']
num_cols = ['Age', 'Avg_Spend_Category', 'Avg_Spend_Region', 'Avg_Spend_Method']

In [22]:
cat_cols

['Gender',
 'Category',
 'Store_Region',
 'Online_Or_Offline',
 'Payment_Method',
 'DayOfWeek',
 'Month',
 'IsWeekend']

In [23]:
num_cols

['Age', 'Avg_Spend_Category', 'Avg_Spend_Region', 'Avg_Spend_Method']

In [24]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (40000, 12)
Test: (10000, 12)


In [26]:
models = [
    ('LinearRegression', LinearRegression()),
    ('DecisionTree', DecisionTreeRegressor(random_state=42)),
    ('RandomForest', RandomForestRegressor(n_estimators=200, max_depth=10,
                                           min_samples_split=5, min_samples_leaf=3, random_state=42)),
    ('GradientBoosting', GradientBoostingRegressor(random_state=42)),
    ('XGBoost', XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbosity=0))
]

In [27]:
results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)


In [28]:
for name, model in models:
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    print(f"Training {name} with 5-Fold Cross-Validation...")
    cv_scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')

    results.append({
        'Model': name,
        'Mean R2': cv_scores.mean(),
        'Std R2': cv_scores.std()
    })

Training LinearRegression with 5-Fold Cross-Validation...
Training DecisionTree with 5-Fold Cross-Validation...
Training RandomForest with 5-Fold Cross-Validation...
Training GradientBoosting with 5-Fold Cross-Validation...
Training XGBoost with 5-Fold Cross-Validation...


In [29]:
results_df = pd.DataFrame(results).sort_values(by='Mean R2', ascending=False)
print("\n📈 Cross-Validation Results:")
display(results_df)


📈 Cross-Validation Results:


,Model,Mean R2,Std R2
0,LinearRegression,-0.000558,0.000199
3,GradientBoosting,-0.002036,0.000948
2,RandomForest,-0.003379,0.001022
4,XGBoost,-0.028627,0.003413
1,DecisionTree,-1.134660,0.028286


### 2

In [32]:
df = pd.read_excel(filepath)

In [33]:

df['Date'] = pd.to_datetime(df['Date'])

df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month_name()
df['IsWeekend'] = df['Date'].dt.weekday >= 5
df['calc_amount'] = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount'])
print(df[['Total_Amount', 'calc_amount']].corr())


              Total_Amount  calc_amount
Total_Amount           1.0          1.0
calc_amount            1.0          1.0


In [34]:
drop_cols = ['Transaction_ID', 'Customer_ID', 'Date', 'calc_amount',
             'Quantity', 'Unit_Price', 'Discount']  # main leakage sources

X = df.drop(columns=drop_cols + ['Total_Amount'])
y = df['Total_Amount']


In [35]:
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()

print("Categorical:", cat_cols)
print("Numerical:", num_cols)


Categorical: ['Gender', 'Category', 'Store_Region', 'Online_Or_Offline', 'Payment_Method', 'DayOfWeek', 'Month']
Numerical: ['Age', 'IsWeekend']


In [36]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


In [37]:
models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=10,
        reg_lambda=15,
        random_state=42,
        verbosity=0
    )
}


In [40]:
cv = GroupKFold(n_splits=5)
groups = df['Customer_ID']

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, groups=groups, scoring='r2')
    results.append([name, scores.mean(), scores.std()])

cv_results = pd.DataFrame(results, columns=['Model', 'Mean R2', 'Std R2'])
print("\n Cross-Validation Results:")
print(cv_results.sort_values(by='Mean R2', ascending=False))



📊 Cross-Validation Results:
              Model   Mean R2    Std R2
0  LinearRegression -0.001077  0.000895
2      RandomForest -0.001875  0.000916
3  GradientBoosting -0.002080  0.001320
1      DecisionTree -0.003530  0.000465
4           XGBoost -0.003669  0.001546


### 3

In [41]:
df = pd.read_excel(filepath)

In [42]:
#  Step 1: Remove direct leakage features
# (They mathematically define Total_Amount)
df = df.drop(columns=['Quantity', 'Unit_Price'])

#  Step 2: Add safe, derived behavioral features
df['Avg_Spend_Per_Category'] = df.groupby('Category')['Total_Amount'].transform('mean')
df['Avg_Spend_Per_Region'] = df.groupby('Store_Region')['Total_Amount'].transform('mean')
df['Avg_Spend_Per_Customer'] = df.groupby('Customer_ID')['Total_Amount'].transform('mean')
df['Customer_Transaction_Count'] = df.groupby('Customer_ID')['Transaction_ID'].transform('count')

# Step 3: Keep useful indirect predictors (Age, Discount, etc.)
# Drop IDs and Date, which don’t help prediction
X = df.drop(columns=['Transaction_ID', 'Customer_ID', 'Date', 'Total_Amount'])
y = df['Total_Amount']

#  Step 4: Identify column types
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("📊 Categorical columns:", cat_cols)
print("🔢 Numerical columns:", num_cols)

#  Step 5: Train-test split (before scaling/encoding)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\n✅ Train/Test shapes:")
print("Train:", X_train.shape)
print("Test:", X_test.shape)

# Step 6: Preprocessing pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

#  Step 7: Transform data for model training / CV
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


📊 Categorical columns: ['Gender', 'Category', 'Store_Region', 'Online_Or_Offline', 'Payment_Method']
🔢 Numerical columns: ['Age', 'Discount', 'Avg_Spend_Per_Category', 'Avg_Spend_Per_Region', 'Avg_Spend_Per_Customer', 'Customer_Transaction_Count']

✅ Train/Test shapes:
Train: (40000, 11)
Test: (10000, 11)


In [43]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    print(f"⏳ Evaluating {name}...")
    scores = cross_val_score(model, X_train_processed, y_train, cv=cv, scoring='r2')
    results.append({
        "Model": name,
        "Mean R2": scores.mean(),
        "Std R2": scores.std()
    })

results_df = pd.DataFrame(results).sort_values(by="Mean R2", ascending=False)
print("\n📊 Cross-Validation Results:")
display(results_df)

⏳ Evaluating LinearRegression...
⏳ Evaluating DecisionTree...
⏳ Evaluating RandomForest...
⏳ Evaluating GradientBoosting...
⏳ Evaluating XGBoost...

📊 Cross-Validation Results:


,Model,Mean R2,Std R2
0,LinearRegression,0.064625,0.004576
3,GradientBoosting,0.063243,0.003758
4,XGBoost,0.061106,0.003333
2,RandomForest,0.060554,0.003568
1,DecisionTree,0.059735,0.002520
